# Pair-bias structural pattern analysis

Measures four properties of `pair_bias` tensors captured from MegaFold's Pairformer stack:

1. **Symmetry** — can we store only the upper triangle?
2. **Spatial sparsity** — are there dead 64x64 tiles to skip?
3. **Temporal reuse** — does block k+1 look like block k?
4. **Cross-head correlation** — do heads carry redundant information?

Each property gets a plot, a summary statistic, and a `STRONG` / `MODERATE` / `WEAK` verdict.

Run the capture first, e.g.:

```
bash scripts/capture_pair_bias_structured.sh
```


## 1. Imports and setup

In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

CAPTURE_PATH = Path("../outputs/pair_bias_structured/run.pt")
print("Capture path:", CAPTURE_PATH.resolve())

## 2. Load and organize captured tensors

Module paths look like `pairformer_stack.layers.<k>.<something>.to_attn_bias`. We parse the
block index and a coarse layer type from the path.

In [ ]:
data = torch.load(CAPTURE_PATH, map_location="cpu")
print(f"Loaded {len(data)} capture points from {CAPTURE_PATH}")


LAYER_PATTERNS = [
    ("pair_bias_attn", re.compile(r"pair_bias")),
    ("tri_attn_starting", re.compile(r"tri_attn_starting|triangle_attention_starting")),
    ("tri_attn_ending", re.compile(r"tri_attn_ending|triangle_attention_ending")),
]


def parse_name(name: str):
    block_match = re.search(r"layers\.(\d+)", name)
    block_idx = int(block_match.group(1)) if block_match else None
    for label, pat in LAYER_PATTERNS:
        if pat.search(name):
            return block_idx, label
    return block_idx, "other"


by_layer: dict[str, dict[int, torch.Tensor]] = defaultdict(dict)
for name, tensor in data.items():
    idx, ltype = parse_name(name)
    if idx is None:
        continue
    by_layer[ltype][idx] = tensor.float()

for ltype, blocks in by_layer.items():
    example = next(iter(blocks.values()))
    print(f"  {ltype}: {len(blocks)} blocks, example shape {tuple(example.shape)}, dtype {example.dtype}")

In [ ]:
def as_hnn(B: torch.Tensor) -> torch.Tensor:
    """Normalize a captured pair-bias tensor to shape [H, N, N]."""
    t = B.float()
    # Drop leading singleton batch dims while 3+ extra axes remain in front of (H, N, N).
    while t.dim() > 3 and t.shape[0] == 1:
        t = t.squeeze(0)
    if t.dim() == 5:
        t = t.reshape(-1, t.shape[-2], t.shape[-1])[: t.shape[-3]]
    if t.dim() == 4:
        t = t.reshape(-1, t.shape[-2], t.shape[-1])
    if t.dim() != 3:
        raise ValueError(f"Expected 3D [H, N, N], got {tuple(B.shape)}")
    return t


sample_ltype = next(iter(by_layer))
sample = next(iter(by_layer[sample_ltype].values()))
print("Sample normalized shape:", tuple(as_hnn(sample).shape))

## 3. Property 1 — Symmetry

$$\text{sym\_ratio}(B) = \frac{\|B_{\text{sym}}\|_F^2}{\|B_{\text{sym}}\|_F^2 + \|B_{\text{anti}}\|_F^2}$$

- STRONG if > 0.9, MODERATE if > 0.7, WEAK otherwise.

In [ ]:
def symmetry_ratio(B: torch.Tensor) -> float:
    t = as_hnn(B)
    if t.shape[-1] != t.shape[-2]:
        return float("nan")  # windowed (atom) attention — skip
    sym = 0.5 * (t + t.transpose(-2, -1))
    anti = 0.5 * (t - t.transpose(-2, -1))
    sym_e = (sym ** 2).sum().item()
    anti_e = (anti ** 2).sum().item()
    denom = sym_e + anti_e
    return sym_e / denom if denom > 0 else float("nan")


fig, ax = plt.subplots(figsize=(10, 4))
sym_summary: dict[str, list[float]] = {}
for ltype, blocks in by_layer.items():
    xs = sorted(blocks.keys())
    ys = [symmetry_ratio(blocks[k]) for k in xs]
    pairs = [(x, y) for x, y in zip(xs, ys) if not np.isnan(y)]
    if not pairs:
        continue
    xs_plot, ys_plot = zip(*pairs)
    ax.plot(xs_plot, ys_plot, "o-", label=ltype, markersize=4)
    sym_summary[ltype] = list(ys_plot)
    print(f"{ltype}: mean={np.mean(ys_plot):.3f}, min={min(ys_plot):.3f}, max={max(ys_plot):.3f}")

ax.axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="no symmetry")
ax.axhline(0.9, color="red", linestyle="--", alpha=0.5, label="strong (>0.9)")
ax.set_xlabel("Pairformer block index")
ax.set_ylabel("Symmetric energy ratio")
ax.set_ylim(0, 1.05)
ax.legend()
ax.set_title("Symmetry of pair_bias across Pairformer blocks")
plt.tight_layout()
plt.show()

## 4. Property 2 — Spatial sparsity (64x64 tiles)

Partition each `[N, N]` into 64x64 tiles, compute per-tile mean absolute value, and count
tiles whose magnitude is below 10% of the median tile. STRONG if mean fraction > 0.30.

In [ ]:
def tile_magnitude_map(B: torch.Tensor, tile_size: int = 64) -> torch.Tensor | None:
    t = as_hnn(B)
    H, N, M = t.shape
    if N != M or N < tile_size:
        return None
    n_tiles = N // tile_size
    Nc = n_tiles * tile_size
    t = t[:, :Nc, :Nc]
    tiles = t.reshape(H, n_tiles, tile_size, n_tiles, tile_size)
    return tiles.abs().mean(dim=(-3, -1))


# Heatmap visualization: attention_pair_bias at middle block, all heads.
target_ltype = "pair_bias_attn" if "pair_bias_attn" in by_layer else next(iter(by_layer))
blocks = by_layer[target_ltype]
mid_idx = sorted(blocks.keys())[len(blocks) // 2]
tm = tile_magnitude_map(blocks[mid_idx])
if tm is not None:
    H = tm.shape[0]
    fig, axes = plt.subplots(1, H, figsize=(3 * H, 3.2), squeeze=False)
    for h in range(H):
        im = axes[0, h].imshow(tm[h].numpy(), cmap="viridis")
        axes[0, h].set_title(f"head {h}")
        plt.colorbar(im, ax=axes[0, h], fraction=0.046)
    plt.suptitle(f"Per-tile |pair_bias| — {target_ltype}, block {mid_idx}, tile=64")
    plt.tight_layout()
    plt.show()
else:
    print(f"{target_ltype} block {mid_idx} is not square (windowed); skipping heatmap.")

# Aggregate low-magnitude tile fractions across blocks (attention_pair_bias only).
sparsity_fracs: list[float] = []
for idx, B in blocks.items():
    tm = tile_magnitude_map(B)
    if tm is None:
        continue
    median = tm.median()
    sparsity_fracs.append((tm < median * 0.1).float().mean().item())

if sparsity_fracs:
    print(
        f"Spatial sparsity — fraction of tiles below 10% of median:\n"
        f"  mean = {np.mean(sparsity_fracs):.2%}\n"
        f"  range = {min(sparsity_fracs):.2%} - {max(sparsity_fracs):.2%}"
    )
else:
    print("No square tensors available for sparsity analysis.")

## 5. Property 3 — Temporal reuse across Pairformer blocks

$$\delta_k = \frac{\|B_{k+1} - B_k\|_F}{\|B_k\|_F}$$

STRONG if median < 0.1, MODERATE if < 0.3, WEAK otherwise.

In [ ]:
def temporal_deltas(block_dict: dict[int, torch.Tensor]) -> list[tuple[int, float]]:
    keys = sorted(block_dict.keys())
    out: list[tuple[int, float]] = []
    for k, k_next in zip(keys[:-1], keys[1:]):
        B_k = as_hnn(block_dict[k])
        B_next = as_hnn(block_dict[k_next])
        if B_k.shape != B_next.shape:
            continue
        base = B_k.norm().item()
        if base == 0:
            continue
        diff = (B_next - B_k).norm().item()
        out.append((k, diff / base))
    return out


fig, ax = plt.subplots(figsize=(10, 4))
temporal_summary: dict[str, list[float]] = {}
for ltype, blocks in by_layer.items():
    deltas = temporal_deltas(blocks)
    if not deltas:
        continue
    xs, ys = zip(*deltas)
    ax.plot(xs, ys, "o-", label=ltype, markersize=4)
    temporal_summary[ltype] = list(ys)
    print(f"{ltype}: mean={np.mean(ys):.3f}, median={np.median(ys):.3f}")

ax.axhline(0.1, color="red", linestyle="--", alpha=0.5, label="strong (<0.1)")
ax.axhline(0.3, color="orange", linestyle="--", alpha=0.5, label="weak (>0.3)")
ax.set_yscale("log")
ax.set_xlabel("Block index k")
ax.set_ylabel("||B_{k+1} - B_k|| / ||B_k||")
ax.set_title("Temporal change of pair_bias across Pairformer blocks")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Property 4 — Cross-head correlation

Flatten each head to a vector, mean-center, L2-normalize, compute `H x H` Pearson
correlation matrix, average across blocks. STRONG if mean off-diagonal > 0.8.

In [ ]:
def head_correlation_matrix(B: torch.Tensor) -> torch.Tensor | None:
    t = as_hnn(B)
    H = t.shape[0]
    flat = t.reshape(H, -1)
    flat = flat - flat.mean(dim=1, keepdim=True)
    norms = flat.norm(dim=1, keepdim=True)
    if (norms == 0).any():
        return None
    normalized = flat / norms
    return normalized @ normalized.T


target_ltype = "pair_bias_attn" if "pair_bias_attn" in by_layer else next(iter(by_layer))
corrs = []
for B in by_layer[target_ltype].values():
    c = head_correlation_matrix(B)
    if c is not None:
        corrs.append(c)

mean_off_corr: float = float("nan")
if corrs:
    # Use min H across blocks to allow stacking.
    H_min = min(c.shape[0] for c in corrs)
    stacked = torch.stack([c[:H_min, :H_min] for c in corrs])
    avg_corr = stacked.mean(dim=0)

    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(avg_corr.numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax)
    ax.set_xlabel("head")
    ax.set_ylabel("head")
    ax.set_title(f"Avg cross-head correlation\n({target_ltype}, {len(corrs)} blocks)")
    plt.tight_layout()
    plt.show()

    H = avg_corr.shape[0]
    mask = 1 - torch.eye(H)
    mean_off_corr = ((avg_corr * mask).sum() / mask.sum()).item()
    print(f"Mean off-diagonal correlation: {mean_off_corr:.3f}")
else:
    print(f"No correlation matrices computed for {target_ltype}.")

## 7. Summary and decision

Combined verdict for the `pair_bias_attn` layer type (the outer Pairformer attention).

In [ ]:
def classify(value: float, strong: float, moderate: float, higher_is_stronger: bool) -> str:
    if np.isnan(value):
        return "N/A"
    if higher_is_stronger:
        if value >= strong:
            return "STRONG"
        if value >= moderate:
            return "MODERATE"
        return "WEAK"
    else:
        if value <= strong:
            return "STRONG"
        if value <= moderate:
            return "MODERATE"
        return "WEAK"


target_ltype = "pair_bias_attn" if "pair_bias_attn" in by_layer else next(iter(by_layer))
blocks = by_layer[target_ltype]

sym_vals = [v for v in (symmetry_ratio(B) for B in blocks.values()) if not np.isnan(v)]
sym_mean = float(np.mean(sym_vals)) if sym_vals else float("nan")

sparsity_vals: list[float] = []
for B in blocks.values():
    tm = tile_magnitude_map(B)
    if tm is None:
        continue
    median = tm.median()
    sparsity_vals.append((tm < median * 0.1).float().mean().item())
sparsity_mean = float(np.mean(sparsity_vals)) if sparsity_vals else float("nan")

delta_vals = [d for _, d in temporal_deltas(blocks)]
delta_median = float(np.median(delta_vals)) if delta_vals else float("nan")


rows = [
    ("Symmetry (energy ratio)",       sym_mean,       "> 0.9",  classify(sym_mean,      0.9,  0.7,  True)),
    ("Spatial sparsity (tile frac)",  sparsity_mean,  "> 0.30", classify(sparsity_mean, 0.30, 0.15, True)),
    ("Temporal reuse (median delta)", delta_median,   "< 0.10", classify(delta_median,  0.10, 0.30, False)),
    ("Cross-head correlation (mean)", mean_off_corr,  "> 0.80", classify(mean_off_corr, 0.80, 0.50, True)),
]

print("=" * 78)
print(f"STRUCTURAL PROPERTY SUMMARY  ({target_ltype}, {len(blocks)} blocks)")
print("=" * 78)
print(f"{'Property':<34}{'Value':>10}   {'Threshold':<10}   Verdict")
print("-" * 78)
for name, value, thresh, verdict in rows:
    value_str = f"{value:.3f}" if not np.isnan(value) else "  N/A"
    print(f"{name:<34}{value_str:>10}   {thresh:<10}   {verdict}")
print("=" * 78)